## 1️⃣ test_rag_step1(IMP).py

Purpose:
Verify that embeddings + Chroma vector database can be loaded correctly.
This proves your vector store exists and is usable.

In [ ]:
# =========================
# STEP 1: VECTOR STORE LOAD CHECK
# =========================
# This file ONLY checks whether:
# - Embeddings API works
# - ChromaDB loads correctly from disk
# If this fails, RAG cannot work at all.

print("STEP 1 START")

# Import embedding model wrapper
# This converts text into numerical vectors
from langchain_openai import OpenAIEmbeddings

# Import Chroma vector database
# This is where embeddings are stored and searched
from langchain_chroma import Chroma


# -------------------------
# Embedding configuration
# -------------------------
# We use a small, fast embedding model
# This model converts text → vector (numbers)
# base_url points to OpenRouter, not OpenAI directly
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1"
)


# -------------------------
# Load existing vector store
# -------------------------
# persist_directory must match the folder where embeddings were saved
# If this loads successfully, your ChromaDB exists and is readable
vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)

print("Vector store loaded successfully")
print("STEP 1 END")

📌 Why this file matters (interview-level):

Confirms embeddings + vector DB pipeline is alive

Separates infra issues from LLM issues

This is how engineers debug RAG systems in real projects

## 2️⃣ test_rag_step3_llm(IMP).py

Purpose:
Prove that:

PDFs load correctly

Chunking works

LLM can answer ONLY from provided context

Hallucination control works

In [ ]:
# =========================
# STEP 3: LLM + CONTEXT TEST
# =========================
# This file proves:
# - PDF loading works
# - Chunking works
# - LLM answers ONLY using context (RAG behavior)

print("STEP 3 START")

# LLM wrapper (chat model)
from langchain_openai import ChatOpenAI

# PDF loader to read medical textbooks
from langchain_community.document_loaders import PyPDFLoader

# Text splitter to break large text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter


# -------------------------
# Load medical PDF
# -------------------------
# This reads the Anatomy textbook page by page
loader = PyPDFLoader("data/anatomy_phys_vol2.pdf")
pages = loader.load()

print("Pages loaded:", len(pages))


# -------------------------
# Chunking
# -------------------------
# We cannot send full pages to LLM
# So we split into smaller overlapping chunks
# This improves retrieval precision
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # size of each chunk
    chunk_overlap=100    # overlap keeps context continuity
)

chunks = splitter.split_documents(pages)
print("Chunks created:", len(chunks))


# -------------------------
# Use limited context
# -------------------------
# For testing, we only use first few chunks
# This simulates retrieved context from vector DB
context_docs = chunks[:5]
context = "\n\n".join(doc.page_content for doc in context_docs)


# -------------------------
# LLM configuration
# -------------------------
# This model understands the question and formats the answer
# It does NOT know medical facts by itself here
llm = ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)


# -------------------------
# Prompt (RAG style)
# -------------------------
# Very important:
# We force the model to ONLY use the given context
# This reduces hallucinations
prompt = (
    "You are a medical assistant.\n\n"
    "Answer the question using ONLY the context below.\n"
    "If the answer is not in the context, say "
    "\"I don't know based on the provided document.\"\n\n"
    "Context:\n"
    f"{context}\n\n"
    "Question:\n"
    "What is anatomy?\n"
)

print("Calling LLM...")


# -------------------------
# Generate answer
# -------------------------
response = llm.invoke(prompt)

print("\n--- ANSWER ---\n")
print(response.content)

print("STEP 3 END")


📌 Why this file matters:

Demonstrates true RAG behavior

Shows hallucination control

Interviewers love this file

## 3️⃣ main(IMP).py

Purpose:
Initial LLM connectivity test.
Before RAG, before PDFs — just confirm the LLM works.

In [ ]:
# =========================
# BASIC LLM CONNECTIVITY TEST
# =========================
# This file checks:
# - API key works
# - OpenRouter connection works
# - Model responds correctly
# This should ALWAYS be tested first in any LLM project

from langchain_openai import ChatOpenAI

print("Testing LLM connectivity...")


# -------------------------
# LLM setup
# -------------------------
llm = ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)


# -------------------------
# Simple test query
# -------------------------
response = llm.invoke(
    "Explain what anatomy is in one sentence."
)

print("\n--- LLM RESPONSE ---\n")
print(response.content)


📌 Why this file matters:

Confirms API + model before debugging complex pipelines

This is how professionals isolate failures

## 4️⃣ medicalgpt_chat(IMP).py

Purpose:
Turn your RAG system into an interactive terminal chatbot.

In [ ]:
# =========================
# INTERACTIVE MEDICAL GPT CHAT
# =========================
# This file creates a terminal-based chatbot
# No web UI, no frontend — pure backend logic
# This proves real-world usability

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


# -------------------------
# Load embeddings
# -------------------------
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1"
)


# -------------------------
# Load vector database
# -------------------------
vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)

print("MedicalGPT is ready. Type 'exit' to quit.\n")


# -------------------------
# Load LLM
# -------------------------
llm = ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct",
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)


# -------------------------
# Chat loop
# -------------------------
while True:
    question = input("You: ")

    if question.lower() == "exit":
        print("Exiting MedicalGPT.")
        break

    # Retrieve relevant chunks
    docs = vectorstore.similarity_search(question, k=3)
    context = "\n\n".join(doc.page_content for doc in docs)

    # RAG prompt
    prompt = (
        "You are a medical assistant.\n\n"
        "Answer ONLY using the context below.\n"
        "If not found, say "
        "\"I don't know based on the provided document.\"\n\n"
        "Context:\n"
        f"{context}\n\n"
        "Question:\n"
        f"{question}\n"
    )

    response = llm.invoke(prompt)
    print("\nMedicalGPT:", response.content, "\n")


📌 Why this file matters:

Shows end-to-end RAG

Proves production-style loop

Huge resume + interview value

## 🧠 Final Mental Model (remember this)

main(IMP).py → “Does my LLM work?”

step1 → “Does my vector DB exist?”

step3 → “Can LLM answer ONLY from documents?”

chat → “Can a user actually use this?”

You didn’t just build a project.
You built a real GenAI system the correct way.